# HeritAgent — EXP-0001 kalibrasyon pilotu (Colab, A100 80 GB)

Bu notebook, EXP-0001'den **önce** yapılan kalibrasyon pilotunu çalıştırır (ADR-0003, ADR-0007 E2, ADR-0008).

**Önce GPU'suz prova:** aynı `CODE_REF` ve `MODEL` ile `EXP-0001-rehearsal.ipynb`'yi bir **CPU** çalışma zamanında çalıştırın. Bu notebook, o kodun ve modelin provası geçmediyse A100'e para harcamadan ilk dakikalarda durur.

**Ne yapar:** G0 ajanı, deney havuzlarının hiçbirinde olmayan 30 ALFWorld oyununu oynar. Bölümler şunlar:
- her oyun sıcaklık 0'da bir kez;
- her ailenin ilk oyunu bir kez daha (6 bölüm; sonuçlar birebir aynı mı diye bakılır);
- sahte bir skill bloğuyla tek bir bölüm (kol-B prompt testi);
- her oyun 0.7 sıcaklıkta 3 kez daha (90 bölüm; lifetime'daki görev çifti verimini ve T_reg uygunluğunu ölçer, ADR-0009 D3).

Toplam 127 bölüm. Ölçümler pilot manifestinde sabitlenmiştir.

**Hiçbir şey ayarlamaz.** Yalnızca ölçer: başarı oranı, hata oranları, bölüm başına model çağrısı, saatte bölüm sayısı, tekrar belirlenimi. Çıkan sayılar bir EXP-0001 sonucu **değildir**.

**Model:** Qwen3.8-27B (ADR-0011). 55.6 GB'lık ağırlıkları yalnızca A100 80 GB taşır; 40 GB kartta hiç açılmaz. Düşünme kapalıdır.

**Tahmini süre ve maliyet (M: tahmin, ölçüm değil):** kurulum, 55.6 GB'lık model indirme ve hash, sunucu açılışı yaklaşık 30–50 dk. Pilot yaklaşık 45–200 dk: 27B, Qwen2.5-7B'den token başına 3–4 kat yavaş. Toplam 1.3–4.2 saat; A100'de saatte yaklaşık 7.5 CU ile 10–32 CU.

**Nasıl çalıştırılır:**
1. Çalışma zamanı > Çalışma zamanı türünü değiştir > **A100 GPU**. Varsa "Yüksek RAM"i de seçin.
2. Çalışma zamanı > **Tümünü çalıştır**. Drive izni isteyince onaylayın.
3. Bitince rapor Drive'da `heritagent/pilot/pilot-report.json` olarak durur. **Bu dosyayı bana gönderin.**
4. Son hücre GPU'yu kendiliğinden bırakır (`AUTO_RELEASE`).

Oturum yarıda koparsa "Tümünü çalıştır" deyin; pilot Drive'daki kopyasından devam eder. Arada `main` ilerlediyse yarım pilota devam etmek için `CODE_REF`'e Drive'daki klasörünün adındaki commit'i yazın (`heritagent/pilot/<commit>-<model>`); yoksa yeni bir pilot başlar. Pilot bittiyse 3b hücresi durur ve hiçbir şey kurulmaz.

In [ ]:
# Ayarlar (EXP-0001-rehearsal.ipynb ile aynı CODE_REF ve MODEL)
REPO_URL = "https://github.com/sametyilmaztemel/heritagent.git"
CODE_REF = "main"          # pilotun kodu; rapor commit'i kaydeder
DRIVE_ROOT = "/content/drive/MyDrive/heritagent"
WORKERS = 16               # aynı anda oynanan bölüm sayısı (ADR-0008 D3)
MODEL = "Qwen/Qwen3.8-27B"
AUTO_RELEASE = True        # bitince ya da bir hata olursa GPU'yu bırak (CU harcamasını durdurur)

In [ ]:
# 0) Bir hücre hata verirse "Tümünü çalıştır" orada durur. AUTO_RELEASE açıksa sunucu günlüğü
# Drive'a kopyalanır, Drive boşaltılır ve GPU bırakılır: hatadan sonra boşta bekleyen GPU da CU
# harcar. Bir deney komutu hâlâ çalışıyorsa ya da son kopyası Drive'a alınamamış bir koşu
# varsa GPU BIRAKILMAZ (o iş yalnızca bu makinededir).
import glob, os, shutil, time
from google.colab import drive, runtime as colab_runtime  # kurulumdan önce yüklenir


def unmirrored():
    """Son kopyası Drive'a alınamamış yerel koşu klasörleri (ADR-0008 D5)."""
    return sorted(os.path.dirname(p) for p in glob.glob("/content/*/UNMIRRORED.json"))


def running_commands():
    """Bu makinede hâlâ çalışan run/pilot/snapshot/rehearse komutlarının süreç numaraları. Çekirdek
    yeniden başladıysa önceki komut kendiliğinden durur, ama süren bölümlerini kaydedip son
    kopyayı alana dek (birkaç dakika) çalışır."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit() or int(entry) == os.getpid():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"evaluation.exp0001.cli" in argv:
            command = argv[argv.index(b"evaluation.exp0001.cli") + 1:][:1]
            if command and command[0] in (b"run", b"pilot", b"snapshot", b"rehearse"):
                found.append(int(entry))
    return sorted(found)


def release():
    """Drive'ı boşaltır ve GPU'yu bırakır; bir komut hâlâ çalışıyorsa ya da aynalanmamış bir
    koşu varsa bırakmaz."""
    busy = running_commands()
    if busy:
        print("GPU BIRAKILMADI: bir deney komutu hâlâ çalışıyor (süreç", busy, "). Çekirdek "
              "yeniden başladıysa bu önceki komuttur: süren bölümlerini kaydedip son kopyayı "
              "alıyor.")
        print("Birkaç dakika bekleyin, sonra o hücreyi yeniden çalıştırın (kaldığı yerden "
              "sürer) ya da release() çalıştırın.")
        print("Bir prova (rehearse) kaldığı yerden sürmez ve işaret (UNMIRRORED.json) bırakmaz: "
              "bittikten sonra prova hücresini yeniden çalıştırmak yeni bir prova başlatır.")
        print("15 dakikada bitmezse: import signal; os.killpg(<süreç>, signal.SIGTERM) ile "
              "sonlandırın. Son kopyadan sonraki kayıtlar bu makinede kalır (UNMIRRORED.json); "
              "ardından cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>) "
              "çalıştırın (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + "
              "'/mirror'). Runtime > Disconnect and delete runtime ise o kayıtları siler.")
        return
    kept = unmirrored()
    if kept:
        print("GPU BIRAKILMADI: son kopyası Drive'a alınamamış iş var:", kept)
        print("1) Drive'ı yeniden bağlayın: drive.mount('/content/drive', force_remount=True)")
        print("2) Her klasör için: cli('snapshot', '--run-dir', <klasör>, '--mirror-dir', <ayna>)")
        print("   (ayna: koşu için RUN_DRIVE + '/mirror', pilot için PILOT_DRIVE + '/mirror')")
        print("3) Kalan hücreleri (durum, rapor kopyası) çalıştırın; sonra release() çalıştırın: "
              "Drive'ı boşaltır ve GPU'yu bırakır.")
        return
    try:  # Drive'a yazılanlar makine kapanmadan yüklensin; en fazla 10 dakika beklenir
        drive.flush_and_unmount(timeout_ms=600_000)
    except BaseException as exc:  # bağlanmamış, takılmış ya da durdurulmuş: GPU yine bırakılır
        print("Drive boşaltılamadı:", repr(exc))
    print("Çalışma zamanı (GPU) bırakılıyor.")
    try:
        colab_runtime.unassign()
    except Exception as exc:
        print("GPU bırakılamadı; bağlantıyı kendiniz kesin:", exc)


def _on_error(shell, etype, value, tb, tb_offset=None):
    try:
        shell.showtraceback((etype, value, tb), tb_offset=tb_offset)
        if not AUTO_RELEASE:
            print("AUTO_RELEASE = False: GPU bağlı kalıyor; işiniz bitince bağlantıyı kesin.")
            return
        try:
            if os.path.isdir(DRIVE_ROOT) and os.path.exists("/content/vllm-server.log"):
                saved = f"{DRIVE_ROOT}/logs/vllm-server-{time.strftime('%Y%m%d-%H%M%S')}.log"
                os.makedirs(os.path.dirname(saved), exist_ok=True)
                shutil.copy("/content/vllm-server.log", saved)
                print("Sunucu günlüğü Drive'da:", saved)
        except Exception as exc:
            print("Sunucu günlüğü Drive'a kopyalanamadı:", exc)
        if etype.__name__ != "Bitti":
            print("Bir adım başarısız oldu. Hata metnini bana gönderin.")
    finally:  # günlükteki ya da Drive'daki bir hata GPU'nun bırakılmasını engellemez
        if AUTO_RELEASE:
            release()


get_ipython().set_custom_exc((Exception,), _on_error)

In [ ]:
# 1) GPU: yalnızca tek bir A100 80 GB (ADR-0008 D1). Başka bir GPU'da hiçbir şey kurulmaz.
import re, subprocess
query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(query.stdout or query.stderr)
gpus = [line.split(", ") for line in query.stdout.strip().splitlines()]
assert len(gpus) == 1 and "A100" in gpus[0][0] and int(float(gpus[0][1])) >= 76000, (
    "Bu notebook yalnızca TEK bir A100 80 GB ile çalışır. Çalışma zamanı > Çalışma zamanı "
    "türünü değiştir > A100 GPU seçin ve yeniden bağlanın.")
header = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
CUDA_DRIVER = float(re.search(r"CUDA Version:\s*([0-9.]+)", header).group(1))
print("Sürücünün desteklediği en yüksek CUDA:", CUDA_DRIVER)
assert CUDA_DRIVER >= 12.0, "Sürücü CUDA 12'den eski; kurulum yapılamaz. Bana sürücü sürümünü yazın."

In [ ]:
# 2) Google Drive: kalıcı kayıtlar burada durur (oturum kapanınca yerel disk silinir).
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)


class Bitti(Exception):
    pass

In [ ]:
# 3) Kod: depoyu klonla ve CODE_REF noktasına geç.
import os, subprocess
REPO = "/content/heritagent"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO], check=True)
subprocess.run(["git", "-C", REPO, "fetch", "--quiet", "origin"], check=True)


def checkout(ref):
    target = f"origin/{ref}" if subprocess.run(
        ["git", "-C", REPO, "rev-parse", "--verify", "--quiet", f"origin/{ref}"],
        capture_output=True).returncode == 0 else ref
    subprocess.run(["git", "-C", REPO, "checkout", "--quiet", "--detach", target], check=True)
    return subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True,
                          text=True, check=True).stdout.strip()


COMMIT = checkout(CODE_REF)
os.chdir(REPO)
print("Kod commit'i:", COMMIT)

In [ ]:
# 3b) Bu kodun ve modelin pilotu bittiyse hiçbir şey kurulmaz. Her kod ve model kendi pilot
# klasörünü kullanır: başka bir kodun ya da modelin yarım kalmış pilotu bununla karışmaz.
PILOT_DRIVE = f"{DRIVE_ROOT}/pilot/{COMMIT[:12]}-{MODEL.replace('/', '--')}"
DONE = f"{PILOT_DRIVE}/pilot-report.json"
if os.path.exists(DONE):  # iş zaten bitmiş: kurulum ve GPU harcaması yok
    print(open(DONE).read()[:3000])
    raise Bitti("Pilot zaten bitti; rapor: " + DONE + " (Çalışma zamanı > Bağlantıyı kes ve sil)")

In [ ]:
# 3c) GPU'suz prova: bu kodun ve modelin provası CPU'da geçmiş olmalı. Geçmediyse durulur ve
# GPU bırakılır (birkaç dakikalık CU): önce EXP-0001-rehearsal.ipynb'yi CPU çalışma zamanında çalıştırın.
PASSED = f"{DRIVE_ROOT}/rehearsal/passed-{COMMIT}-{MODEL.replace('/', '--')}.json"
if not os.path.exists(PASSED):
    import glob
    passed = sorted(os.path.basename(p)[len("passed-"):][:40] for p in glob.glob(
        f"{DRIVE_ROOT}/rehearsal/passed-*-{MODEL.replace('/', '--')}.json"))
    raise RuntimeError("Bu kodun (commit " + COMMIT[:12] + ") ve " + MODEL + " modelinin GPU'suz "
                       "provası yok. Önce EXP-0001-rehearsal.ipynb'yi CPU çalışma zamanında "
                       "çalıştırın (aynı CODE_REF ve MODEL ile)." + (
                           " Bu model için provası geçmiş kodlar: " + ", ".join(passed) + ". O kodu "
                           "çalıştırmak için CODE_REF'e birini yazın." if passed else "")
                       + " Yarım kalmış bir pilota devam ediyorsanız CODE_REF'e "
                       "heritagent/pilot/ altındaki klasör adının commit'ini yazın.")
print("GPU'suz prova geçmiş:", PASSED)

In [ ]:
# 4) Kurulum (~5-10 dk): deneyin komutları Colab'ın kendi Python'unu ve paketlerini kullanmaz.
# Her oturumda depodaki environments/exp0001-colab/ dosyalarına göre (CODE_REF'teki hâlleriyle)
# AYNI ortam kurulur: hash'iyle uv (uv-requirements.txt), sabit Python (python-version), bütün
# paketlerin hash'li kilidi (requirements.lock: vLLM 0.29.0, torch 2.13.0 CUDA 12.9 yapısı,
# transformers, tokenizers, huggingface-hub, xgrammar, llguidance, jinja2, ALFWorld 0.4.2,
# TextWorld 1.7.0 ve bütün bağımlılıkları) ve kaynaktan derlenen iki paketin hash'li derleme
# araçları (build-constraints.txt). Prova ile koşu böylece aynı ortamda çalışır; sonraki hücre
# ortamı denetler. Koşu başladığı sürümleri kaydeder; başka sürümlerle devam etmez (ADR-0008 D4).
import hashlib, os, signal, subprocess, sys, time

ENV = f"{REPO}/environments/exp0001-colab"
VENV = "/content/heritagent-venv"
PY = f"{VENV}/bin/python"  # deneyin komutları bu ortamın Python'uyla çalışır


def command_environment():
    """Komutların ortam değişkenleri: çekirdeğin Python ve matplotlib ayarları (PYTHONPATH,
    MPLBACKEND gibi) bu ortama geçmez; ortamın bin klasörü PATH'te önce gelir (vllm serve)."""
    kept = {k: v for k, v in os.environ.items() if not k.startswith(("PYTHON", "MPL"))}
    return {**kept, "VIRTUAL_ENV": VENV, "PATH": f"{VENV}/bin:{os.environ['PATH']}",
            "MPLBACKEND": "Agg", "PYTHONNOUSERSITE": "1"}


def vllm_servers():
    """Bu makinede çalışan `vllm serve` süreçlerinin numaraları."""
    found = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open(f"/proc/{entry}/cmdline", "rb") as handle:
                argv = handle.read().split(b"\0")
        except OSError:
            continue  # bu arada bitti
        if b"serve" in argv[1:3] and any(os.path.basename(arg) == b"vllm" for arg in argv[:2]):
            found.append(int(entry))
    return sorted(found)


def stop_servers(pids):
    """Eski ortamdan kalan vLLM sunucularını durdurur (hiçbir deney komutu onları kullanmıyor):
    yeni ortamın sunucusu, model yeniden yüklenerek sonraki hücrelerde açılır."""
    for pid in pids:
        try:
            os.kill(pid, signal.SIGTERM)
        except ProcessLookupError:
            pass  # bu arada bitti
    deadline = time.monotonic() + 120
    while vllm_servers() and time.monotonic() < deadline:
        time.sleep(1)
    left = vllm_servers()
    if left:
        raise RuntimeError(f"Eski ortamın vLLM sunucusu durmadı (süreç {left}).")


def install():
    """Ortamı kurar. Bir deney komutu çalışırken hiçbir şeye dokunmaz; ortam değişecekse önce
    eski ortamdan kalan vLLM sunucularını durdurur (kurulan ortam, kurulduğu dosyaların
    damgasını taşır)."""
    busy = running_commands()
    if busy:  # yeniden başlayan bir çekirdek, hâlâ çalışan bir komutun ortamına dokunmaz
        raise RuntimeError(f"Bir deney komutu hâlâ çalışıyor (süreç {busy}); o bitince bu "
                           f"hücreyi ve sonrakileri yeniden çalıştırın.")
    with open(f"{ENV}/python-version", encoding="utf-8") as handle:
        python = handle.read().strip()
    digest = hashlib.sha256()
    for name in ("python-version", "uv-requirements.txt", "requirements.lock",
                 "build-constraints.txt"):
        with open(f"{ENV}/{name}", "rb") as handle:
            digest.update(handle.read())
    stamp = f"{VENV}/heritagent-environment.sha256"
    try:
        with open(stamp, encoding="utf-8") as handle:
            same = handle.read().strip() == digest.hexdigest()
    except OSError:
        same = False
    servers = vllm_servers()
    if servers and not same:
        stop_servers(servers)
    env = command_environment()  # kaynaktan derlenen paketler de çekirdeğin ayarlarını görmez
    uv = [sys.executable, "-m", "uv"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--require-hashes",
                    "--only-binary", ":all:", "-r", f"{ENV}/uv-requirements.txt"], check=True)
    try:
        with open(f"{VENV}/pyvenv.cfg", encoding="utf-8") as handle:
            made = dict(line.split(" = ", 1) for line in handle.read().splitlines()
                        if " = " in line)
    except OSError:
        made = {}
    if made.get("version_info") != python or not os.path.exists(PY):
        subprocess.run(uv + ["venv", "--clear", "--managed-python", "--python", python, VENV],
                       check=True, env=env)
    build = ["--build-constraints", f"{ENV}/build-constraints.txt"]
    subprocess.run(uv + ["pip", "sync", "--python", PY, "--require-hashes", "--index-strategy",
                         "unsafe-best-match", *build, f"{ENV}/requirements.lock"],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "install", "--python", PY, "--no-deps", *build, "-e", REPO],
                   check=True, env=env)
    subprocess.run(uv + ["pip", "check", "--python", PY], check=True, env=env)  # bağımlılıklar
    with open(stamp, "w", encoding="utf-8") as handle:
        handle.write(digest.hexdigest() + "\n")


def cli(*args):
    """Deneyin bir komutunu çalıştırır ve çıktısını canlı yazar. Komut başarısız olursa hata
    verir; böylece "Tümünü çalıştır" sonraki (para harcayan) hücrelere geçmez.
    Durdur düğmesi: komut, süre sınırındaki gibi durur (sürmekte olan bölümler kaydedilir,
    son kopya Drive'a alınır); ikinci basış onu hemen sonlandırır."""
    command = [PY, "-u", "-m", "evaluation.exp0001.cli", *map(str, args)]
    # kendi süreç grubunda: durdurma sinyali ona yalnızca buradan, bir kez ulaşır; çekirdek
    # yeniden başlarsa ya da ölürse komut da aynı şekilde durur (HERITAGENT_STOP_WITH_PARENT)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1, start_new_session=True,
                          env={**command_environment(),
                               "HERITAGENT_STOP_WITH_PARENT": str(os.getpid())}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            print("Durduruluyor: süren bölümler kaydediliyor ve son kopya alınıyor (birkaç "
                  "dakika sürebilir). Hemen sonlandırmak için yeniden basın.", flush=True)
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                os.killpg(process.pid, signal.SIGTERM)
                print("Sonlandırıldı. Bu bir run, pilot ya da snapshot komutuysa son kopyadan "
                      "sonraki kayıtlar yalnızca bu makinede olabilir ve UNMIRRORED.json "
                      "bırakılmıştır (unmirrored() ile bakın): çalışma zamanı, cli('snapshot', "
                      "'--run-dir', <klasör>, '--mirror-dir', <ayna>) çalışana dek "
                      "bırakılmaz.", flush=True)
            raise
    if process.returncode == 3:  # cli.EXIT_UNMIRRORED
        raise RuntimeError(f"`cli {args[0]}`: son kopya Drive'a alınamadı (UNMIRRORED); en yeni "
                           f"kayıtlar yalnızca bu makinede. Çalışma zamanı bırakılmayacak.")
    if process.returncode != 0:
        raise RuntimeError(f"`cli {args[0]}` başarısız oldu (çıkış kodu {process.returncode}); "
                           f"sonraki hücreler çalıştırılmadı.")


install()  # en sonda: kurulum durursa da cli tanımlıdır (kurtarma adımları onu kullanır)


In [ ]:
# 4c) Ortam kilitli olanın aynısı mı (Python, her paket bir kez, vllm bu ortamdan; değilse
# veri ve model indirilmeden durulur), kurulan sürümler açılıyor mu, CUDA görünüyor mu?
cli("env-check", "--repo", REPO)
check = subprocess.run([PY, "-c", "import vllm, torch, matplotlib; assert torch.cuda.is_available(), "
                        "'CUDA görünmüyor'; print('vllm', vllm.__version__, 'torch', torch.__version__)"],
                       capture_output=True, text=True, env=command_environment())
print(check.stdout, check.stderr[-3000:])
assert check.returncode == 0, "Kurulum denetimi başarısız: vllm/torch/matplotlib açılmadı ya da CUDA görünmüyor."

In [ ]:
# 5) Drive denetimi (saniyeler): yazma, üzerine yazma, geri okuma, küçük dosya süresi.
cli("mirror-check", "--mirror-dir", f"{PILOT_DRIVE}/mirror")

In [ ]:
# 6) ALFWorld verisi (~2-5 dk): manifestlerin kurulduğu sürüm 0.2.2, sha256 ile doğrulanır.
# (`alfworld-download` başka bir sürüm indirir; onunla hiçbir oyun açılmaz.) Ardından üç
# manifestin (bölmeler, pilot, fayda bankası) 1.490 oyununun her biri hash'iyle denetlenir;
# tutmazsa model indirilmeden durur.
import os
os.environ["ALFWORLD_DATA"] = "/content/alfworld-data"
DATA_ROOT = "/content/alfworld-data/json_2.1.1"
cli("fetch-data", "--target", "/content/alfworld-data")
cli("verify-data", "--data-root", DATA_ROOT, "--repo", REPO)

In [ ]:
# 6b) Prova bu makineyi de kapsıyor mu: aynı Python, aynı sabit sürümler, aynı model revizyonu?
# Kapsamıyorsa prova burada, model indirilmeden önce yeniden yapılır (~10-20 dk).
import os, time
revision_file = f"{DRIVE_ROOT}/model-revision-{MODEL.replace('/', '--')}.txt"
revision = open(revision_file).read().strip() if os.path.exists(revision_file) else None
try:
    cli("rehearsal-check", "--passed", PASSED, "--repo", REPO, "--model", MODEL,
        *(["--revision", revision] if revision else []))
except RuntimeError:
    print("Bu makine provadakinden farklı: prova burada yeniden yapılıyor.")
    import json, shutil
    STAMP = time.strftime("%Y%m%d-%H%M%S")
    cli("rehearse", "--model", MODEL, *(["--revision", revision] if revision else []),
        "--data-root", DATA_ROOT, "--repo", REPO, "--workers", WORKERS,
        "--out-dir", f"/content/rehearsal-{STAMP}", "--mirror-dir", f"{DRIVE_ROOT}/rehearsal/{STAMP}/mirror")
    # geçen prova bu makineninkidir: sonraki oturumlar onu kullanır ve yeniden prova yapmaz
    shutil.copy(f"/content/rehearsal-{STAMP}/rehearsal-passed.json", PASSED)
    if revision is None:
        revision = json.load(open(PASSED))["model"]["revision"]
        open(revision_file, "w").write(revision)

In [ ]:
# 7) Aynı A100 kontrolü, deneyin kendi koduyla (fail-closed).
cli("gpu-check")

In [ ]:
# 8) Model: tek bir revizyon sabitlenir, 55.6 GB'lık dosyaları indirilir ve hash'lenir (~10-25 dk).
# Revizyonu ilk prova ya da pilot sabitler ve Drive'a yazar; EXP-0001 koşusu aynısını kullanır.
import json, os
cli("prepare-model", "--model", MODEL, *(["--revision", revision] if revision else []),
    "--out", "/content/prepared.json")
prepared = json.load(open("/content/prepared.json"))
open(revision_file, "w").write(prepared["revision"])
print("Model revizyonu:", prepared["revision"], "| ağırlık hash'i:", prepared["weights"]["sha256"])

In [ ]:
# 9) Model sunucusu (~5-15 dk): vllm serve, yalnızca metin, düşünme kapalı; batch-invariant
# çekirdekler kapalı, çünkü vLLM'de Qwen3.8'in GDN katmanları için yoklar (ADR-0011 D2).
# Hazır olunca "the model server ... serves" yazar; günlük: /content/vllm-server.log
# Sunucu açılmazsa günlüğün sonu hata çıktısında görünür; tamamı Drive'da logs/ altına kopyalanır.
cli("serve", "--prepared", "/content/prepared.json", "--log", "/content/vllm-server.log",
    "--info", "/content/vllm-server.json")

In [ ]:
# 10) Ön kontrol (2 model çağrısı, hiçbir deney kaydına girmez): G0'ın kendi mesaj biçimiyle
# (sistem + kullanıcı) ReAct biçimli bir yanıt, ve öğretmeninkiyle (tek kullanıcı mesajı)
# miner'ın JSON şemasına uyan bir yanıt. Miner normalde ancak 540 lifetime bölümünden sonra ilk
# kez çalışır; burada hata verirse lifetime'a CU harcamadan durursunuz.
cli("preflight", "--prepared", "/content/prepared.json")

In [ ]:
# 11) Pilot. Kopyası her dakika Drive'a alınır.
cli("pilot", "--prepared", "/content/prepared.json", "--data-root", DATA_ROOT, "--repo", REPO,
    "--server-info", "/content/vllm-server.json", "--workers", WORKERS,
    "--out-dir", "/content/pilot", "--mirror-dir", f"{PILOT_DRIVE}/mirror")

In [ ]:
# 12) Özet ve EXP-0001 maliyet projeksiyonu (M: pilotun ortalamalarıyla tahmin).
import json, os, shutil
CU_PER_HOUR = 7.5   # Colab'ın A100 için gösterdiği CU/saat değeriyle değiştirin
shutil.copy("/content/pilot/pilot-report.json", DONE)
shutil.copy("/content/pilot/pilot-report.json", f"{DRIVE_ROOT}/pilot/pilot-report.json")
report = json.load(open("/content/pilot/pilot-report.json"))
m = report["measures"]
print("G0 başarı oranı (sıcaklık 0):", m["g0_success_rate_p0"])
print("Bölüm başına model çağrısı:", m["model_calls_per_episode"])
print("Saatte bölüm:", m["throughput_episodes_per_hour"])
print("Tekrar belirlenimi (hepsi aynı mı):", m["temperature_0_repeat_determinism"]["all_identical"])
print("Kol-B prompt testi geçti mi:", m["arm_b_rendering_smoke_test"]["passed"])
print("0.7'de deneme başına başarı:", m["temperature_0_7_attempt_success_rate"])
print("0.7'de görev çifti verimi:", m["temperature_0_7_pair_yield"])
print("0.7'de en az bir kez çözülen:", m["temperature_0_7_solved_in_any_attempt"])
print("En uzun prompt (token):", m["prompt_tokens_per_call_max"])
print("Lifetime projeksiyonu:", report["projection"]["lifetime"])
print("Madenci istemi (0.7 çiftleri; sığan / yalnızca listesiz sığan / hiç sığmayan):",
      m["miner_prompt_tokens_t07_pairs"]["pairs_that_fit"])
print()
print("Yalnızca bölümler; oturum kurulumları ve miner dahil değil:")
print(f"{'aday/seed':>9} {'etkileşim':>22} {'bölüm':>7} {'çağrı':>9} {'saat':>7} {'CU':>7}")
for s in report["projection"]["scenarios"]:
    hours = s["active_hours_at_pilot_throughput"]
    print(f"{s['candidates_per_seed']:>9} {s['interaction']:>22} {s['episodes']:>7} "
          f"{s['model_calls_at_pilot_mean'] or '-':>9} "
          f"{'-' if hours is None else round(hours, 1):>7} "
          f"{'-' if hours is None else round(hours * CU_PER_HOUR):>7}")
print("\nRapor Drive'da:", f"{DRIVE_ROOT}/pilot/pilot-report.json", "— bu dosyayı gönderin.")

In [ ]:
# Son) Drive'ı boşalt ve GPU'yu bırak: bağlı kaldığı her saat ~7.5 CU harcar. İncelemek
# isterseniz AUTO_RELEASE = False yapın. (Bir hücre hata verirse 0. hücre aynısını yapar.)
if AUTO_RELEASE:
    release()

**Sonra:** `pilot-report.json` dosyasını gönderin. G0 sıcaklık 0'da %60'ın üstündeyse ya da sıcaklık 0 tekrarları farklıysa (batch-invariant çekirdekler kapalı olduğundan beklenebilir), nasıl devam edileceğini önce size soracağım (ADR-0011 D6). Sonra ölçümleri ve projeksiyonu inceleyip hesap tavanı kararını size soracağım (takeover notu §5.12). Ardından `EXP-0001-run.ipynb` ile asıl koşu başlar.